## TUAB (TUH ABNORMAL EEG) PREPROCESSING

In [1]:
# !pip install mne

In [2]:
# IMPORTS
import numpy as np
import mne
import h5py
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

mne.set_log_level('ERROR')

print("✓ Imports loaded")

✓ Imports loaded


In [3]:
# CONFIGURATION
@dataclass
class TUABConfig:
    """Configuration for TUAB preprocessing."""
    
    # Paths 
    RAW_DIR: Path = Path('data/raw/tuh_eeg_abnormal')
    PROCESSED_DIR: Path = Path('data/processed')
    
    # Preprocessing parameters (match CHB-MIT pipeline)
    SFREQ_TARGET: int = 100          # Resample to 100 Hz
    L_FREQ: float = 0.5              # High-pass filter
    H_FREQ: float = 35.0             # Low-pass filter
    NOTCH_FREQ: float = 60.0         # US line noise (60 Hz)
    
    # Epoch parameters
    EPOCH_DURATION: float = 30.0     # seconds (match Sleep-EDF/CHB-MIT)
    
    # Quality thresholds
    MIN_DURATION: float = 60.0       # Minimum file duration (seconds)
    AMPLITUDE_THRESHOLD: float = 500.0  # µV, for artifact rejection
    
    # Standard 10-20 channels we want to extract
    TARGET_CHANNELS: List[str] = None
    
    def __post_init__(self):
        self.TARGET_CHANNELS = [
            'FP1', 'FP2', 'F7', 'F3', 'FZ', 'F4', 'F8',
            'T3', 'C3', 'CZ', 'C4', 'T4',
            'T5', 'P3', 'PZ', 'P4', 'T6',
            'O1', 'O2'
        ]  # 19 channels
        
        # Channel name aliases (TUH uses various naming conventions)
        self.CHANNEL_ALIASES = {
            # T7/T8 naming convention
            'T7': 'T3', 'T8': 'T4', 'P7': 'T5', 'P8': 'T6',
            # With -REF suffix
            'FP1-REF': 'FP1', 'FP2-REF': 'FP2',
            'F7-REF': 'F7', 'F3-REF': 'F3', 'FZ-REF': 'FZ', 'F4-REF': 'F4', 'F8-REF': 'F8',
            'T3-REF': 'T3', 'C3-REF': 'C3', 'CZ-REF': 'CZ', 'C4-REF': 'C4', 'T4-REF': 'T4',
            'T5-REF': 'T5', 'P3-REF': 'P3', 'PZ-REF': 'PZ', 'P4-REF': 'P4', 'T6-REF': 'T6',
            'O1-REF': 'O1', 'O2-REF': 'O2',
            'T7-REF': 'T3', 'T8-REF': 'T4', 'P7-REF': 'T5', 'P8-REF': 'T6',
            # With EEG prefix
            'EEG FP1-REF': 'FP1', 'EEG FP2-REF': 'FP2',
            'EEG F7-REF': 'F7', 'EEG F3-REF': 'F3', 'EEG FZ-REF': 'FZ',
            'EEG F4-REF': 'F4', 'EEG F8-REF': 'F8',
            'EEG T3-REF': 'T3', 'EEG C3-REF': 'C3', 'EEG CZ-REF': 'CZ',
            'EEG C4-REF': 'C4', 'EEG T4-REF': 'T4',
            'EEG T5-REF': 'T5', 'EEG P3-REF': 'P3', 'EEG PZ-REF': 'PZ',
            'EEG P4-REF': 'P4', 'EEG T6-REF': 'T6',
            'EEG O1-REF': 'O1', 'EEG O2-REF': 'O2',
            'EEG T7-REF': 'T3', 'EEG T8-REF': 'T4',
            'EEG P7-REF': 'T5', 'EEG P8-REF': 'T6',
            # LE (Linked Ear) reference
            'FP1-LE': 'FP1', 'FP2-LE': 'FP2',
            'F7-LE': 'F7', 'F3-LE': 'F3', 'FZ-LE': 'FZ', 'F4-LE': 'F4', 'F8-LE': 'F8',
            'T3-LE': 'T3', 'C3-LE': 'C3', 'CZ-LE': 'CZ', 'C4-LE': 'C4', 'T4-LE': 'T4',
            'T5-LE': 'T5', 'P3-LE': 'P3', 'PZ-LE': 'PZ', 'P4-LE': 'P4', 'T6-LE': 'T6',
            'O1-LE': 'O1', 'O2-LE': 'O2',
        }

config = TUABConfig()
print(f"✓ Config loaded")
print(f"  Raw dir: {config.RAW_DIR}")
print(f"  Output dir: {config.PROCESSED_DIR}")

✓ Config loaded
  Raw dir: data/raw/tuh_eeg_abnormal
  Output dir: data/processed


In [4]:
# FILE DISCOVERY
def discover_tuab_files(raw_dir: Path) -> Dict[str, List[Dict]]:
    """
    Discover all TUAB EDF files and their labels.
    
    TUAB structure:
    raw_dir/
    ├── eval/
    │   ├── normal/01_tcp_ar/*.edf
    │   └── abnormal/01_tcp_ar/*.edf
    └── train/
        ├── normal/01_tcp_ar/*.edf
        └── abnormal/01_tcp_ar/*.edf
    
    Returns:
        Dict with 'eval' and 'train' keys, each containing list of file info dicts
    """
    files = {'eval': [], 'train': []}
    
    for split in ['eval', 'train']:
        split_dir = raw_dir / split
        if not split_dir.exists():
            print(f"  Warning: {split} directory not found")
            continue
        
        for label_name in ['normal', 'abnormal']:
            label = 0 if label_name == 'normal' else 1
            label_dir = split_dir / label_name
            
            if not label_dir.exists():
                continue
            
            # Find all EDF files (may be in subdirectories like 01_tcp_ar)
            edf_files = list(label_dir.rglob('*.edf'))
            
            for edf_path in edf_files:
                # Extract subject ID from filename (e.g., aaaaamye_s001_t000.edf)
                subject_id = edf_path.stem.split('_')[0]
                
                files[split].append({
                    'path': edf_path,
                    'label': label,
                    'label_name': label_name,
                    'subject_id': subject_id,
                    'split': split
                })
    
    # Summary
    for split in ['eval', 'train']:
        if files[split]:
            n_normal = sum(1 for f in files[split] if f['label'] == 0)
            n_abnormal = sum(1 for f in files[split] if f['label'] == 1)
            print(f"  {split}: {len(files[split])} files ({n_normal} normal, {n_abnormal} abnormal)")
    
    return files


# Discover files
print("\nDiscovering TUAB files...")
tuab_files = discover_tuab_files(config.RAW_DIR)


Discovering TUAB files...
  eval: 276 files (150 normal, 126 abnormal)
  train: 2717 files (1371 normal, 1346 abnormal)


In [5]:
# PREPROCESSING CLASS
class TUABPreprocessor:
    """Preprocess TUAB EEG files."""
    
    def __init__(self, config: TUABConfig):
        self.config = config
    
    def standardize_channel_name(self, ch_name: str) -> Optional[str]:
        """Map various channel naming conventions to standard names."""
        # Clean up the name
        ch_upper = ch_name.upper().strip()
        
        # Check aliases
        if ch_upper in self.config.CHANNEL_ALIASES:
            return self.config.CHANNEL_ALIASES[ch_upper]
        
        # Check if it's already a target channel
        if ch_upper in self.config.TARGET_CHANNELS:
            return ch_upper
        
        # Try removing common prefixes/suffixes
        for prefix in ['EEG ', 'EEG-']:
            if ch_upper.startswith(prefix):
                ch_clean = ch_upper[len(prefix):]
                if ch_clean in self.config.TARGET_CHANNELS:
                    return ch_clean
                if ch_clean in self.config.CHANNEL_ALIASES:
                    return self.config.CHANNEL_ALIASES[ch_clean]
        
        return None
    
    def process_file(self, file_info: Dict) -> Optional[Dict]:
        """
        Process a single TUAB EDF file.
        
        Returns:
            Dict with 'epochs', 'labels', 'subject_id' or None if failed
        """
        edf_path = file_info['path']
        label = file_info['label']
        subject_id = file_info['subject_id']
        
        try:
            # Load EDF
            raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
            
            # Check duration
            duration = raw.times[-1]
            if duration < self.config.MIN_DURATION:
                return None
            
            # Map channels to standard names
            ch_mapping = {}
            available_channels = []
            
            for ch in raw.ch_names:
                std_name = self.standardize_channel_name(ch)
                if std_name and std_name in self.config.TARGET_CHANNELS:
                    if std_name not in available_channels:  # Avoid duplicates
                        ch_mapping[ch] = std_name
                        available_channels.append(std_name)
            
            # Need at least 10 channels
            if len(available_channels) < 10:
                return None
            
            # Pick and rename channels
            raw.pick_channels(list(ch_mapping.keys()))
            raw.rename_channels(ch_mapping)
            
            # Reorder to standard order (pad missing with zeros later)
            present_channels = [ch for ch in self.config.TARGET_CHANNELS if ch in raw.ch_names]
            raw.reorder_channels(present_channels)
            
            # Preprocessing pipeline
            # 1. Notch filter (60 Hz US line noise)
            raw.notch_filter(self.config.NOTCH_FREQ, verbose=False)
            
            # 2. Bandpass filter
            raw.filter(self.config.L_FREQ, self.config.H_FREQ, verbose=False)
            
            # 3. Resample to target frequency
            if raw.info['sfreq'] != self.config.SFREQ_TARGET:
                raw.resample(self.config.SFREQ_TARGET, verbose=False)
            
            # 4. Create epochs
            n_samples_per_epoch = int(self.config.EPOCH_DURATION * self.config.SFREQ_TARGET)
            data = raw.get_data()  # (n_channels, n_samples)
            
            n_epochs = data.shape[1] // n_samples_per_epoch
            if n_epochs == 0:
                return None
            
            # Reshape into epochs
            epochs_data = data[:, :n_epochs * n_samples_per_epoch]
            epochs_data = epochs_data.reshape(data.shape[0], n_epochs, n_samples_per_epoch)
            epochs_data = epochs_data.transpose(1, 0, 2)  # (n_epochs, n_channels, n_samples)
            
            # 5. Pad to 19 channels if needed
            n_channels = epochs_data.shape[1]
            if n_channels < 19:
                pad_shape = (epochs_data.shape[0], 19 - n_channels, epochs_data.shape[2])
                padding = np.zeros(pad_shape)
                epochs_data = np.concatenate([epochs_data, padding], axis=1)
            
            # 6. Z-score normalization (per epoch, per channel)
            for i in range(epochs_data.shape[0]):
                for j in range(epochs_data.shape[1]):
                    ch_data = epochs_data[i, j, :]
                    if ch_data.std() > 0:
                        epochs_data[i, j, :] = (ch_data - ch_data.mean()) / ch_data.std()
            
            # 7. Artifact rejection
            max_amp = np.abs(epochs_data).max(axis=(1, 2))
            good_epochs = max_amp < self.config.AMPLITUDE_THRESHOLD
            epochs_data = epochs_data[good_epochs]
            
            if len(epochs_data) == 0:
                return None
            
            # Create labels array (same label for all epochs from this file)
            labels = np.full(len(epochs_data), label, dtype=np.int64)
            subject_ids = [subject_id] * len(epochs_data)
            
            return {
                'epochs': epochs_data.astype(np.float32),
                'labels': labels,
                'subject_ids': subject_ids
            }
            
        except Exception as e:
            return None
    
    def process_split(
        self, 
        files: List[Dict], 
        max_files: Optional[int] = None,
        desc: str = "Processing"
    ) -> Dict:
        """Process all files in a split."""
        
        all_epochs = []
        all_labels = []
        all_subject_ids = []
        
        files_to_process = files[:max_files] if max_files else files
        
        successful = 0
        for file_info in tqdm(files_to_process, desc=desc):
            result = self.process_file(file_info)
            
            if result is not None:
                all_epochs.append(result['epochs'])
                all_labels.append(result['labels'])
                all_subject_ids.extend(result['subject_ids'])
                successful += 1
        
        if not all_epochs:
            return None
        
        # Combine
        combined = {
            'epochs': np.concatenate(all_epochs, axis=0),
            'labels': np.concatenate(all_labels, axis=0),
            'subject_ids': all_subject_ids,
            'sfreq': self.config.SFREQ_TARGET,
            'n_channels': 19,
            'n_samples': int(self.config.EPOCH_DURATION * self.config.SFREQ_TARGET),
            'n_files_processed': successful,
            'n_files_total': len(files_to_process)
        }
        
        print(f"\n  Processed {successful}/{len(files_to_process)} files")
        print(f"  Total epochs: {len(combined['epochs'])}")
        print(f"  Class distribution: Normal={sum(combined['labels']==0)}, Abnormal={sum(combined['labels']==1)}")
        
        return combined

In [6]:
# SAVE TO HDF5
def save_tuab_to_hdf5(data: Dict, output_path: Path):
    """Save processed TUAB data to HDF5 (same format as CHB-MIT)."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        with h5py.File(output_path, 'w') as f:
            f.create_dataset('epochs', data=data['epochs'], compression='gzip', compression_opts=4)
            f.create_dataset('labels', data=data['labels'], compression='gzip')
            
            subject_ids = np.array(data['subject_ids'], dtype='S20')
            f.create_dataset('subject_ids', data=subject_ids)
            
            f.attrs['sfreq'] = data['sfreq']
            f.attrs['n_channels'] = data['n_channels']
            f.attrs['n_samples'] = data['n_samples']
            f.attrs['epoch_duration'] = 30.0
            f.attrs['dataset'] = 'tuab'
            f.attrs['task'] = 'abnormal_detection'
            f.attrs['n_classes'] = 2
            f.attrs['class_names'] = str(['Normal', 'Abnormal'])
        
        print(f"\n✓ Saved to {output_path}")
        print(f"  Shape: {data['epochs'].shape}")
        print(f"  Size: {output_path.stat().st_size / 1e6:.1f} MB")
        
    except Exception as e:
        print(f"\n✗ Error saving: {e}")
        if output_path.exists():
            output_path.unlink()
        raise e

In [7]:
# MAIN PROCESSING FUNCTION
# ROBUST VERSION - saves as it goes, survives kernel restarts

def process_tuab_robust(
    raw_dir: Path = None,
    output_dir: Path = None,
    max_files_eval: Optional[int] = None,
    max_files_train: Optional[int] = None,
    batch_size: int = 200  # Save every 200 files
):
    """Process TUAB with incremental saving - survives kernel restarts."""
    raw_dir = raw_dir or config.RAW_DIR
    output_dir = Path(output_dir or config.PROCESSED_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print("=" * 60)
    print("TUAB PREPROCESSING (ROBUST)")
    print("=" * 60)
    
    files = discover_tuab_files(raw_dir)
    preprocessor = TUABPreprocessor(config)
    
    for split in ['eval', 'train']:
        max_files = max_files_eval if split == 'eval' else max_files_train
        split_files = files[split]
        
        if not split_files:
            continue
        if max_files == 0:
            continue
            
        split_files = split_files[:max_files] if max_files else split_files
        
        print(f"\nProcessing {split} ({len(split_files)} files)...")
        
        # Check what's already saved (resume support)
        existing_chunks = sorted(output_dir.glob(f'tuab_{split}_chunk*.h5'))
        already_processed = 0
        
        for chunk_file in existing_chunks:
            try:
                with h5py.File(chunk_file, 'r') as f:
                    already_processed += f.attrs.get('files_in_chunk', 0)
                print(f" Found existing {chunk_file.name}")
            except:
                chunk_file.unlink()
                print(f"  ⚠ Removed corrupted {chunk_file.name}")
        
        if already_processed >= len(split_files):
            print(f" {split} already fully processed!")
            continue
        
        # Skip already-processed files
        chunk_idx = len(existing_chunks)
        files_remaining = split_files[already_processed:]
        print(f"  Resuming from file {already_processed}, chunk {chunk_idx}")
        
        # Process in batches
        batch_epochs, batch_labels, batch_sids = [], [], []
        files_in_batch = 0
        successful = already_processed
        
        for file_info in tqdm(files_remaining, desc=f"{split}"):
            result = preprocessor.process_file(file_info)
            
            if result is not None:
                batch_epochs.append(result['epochs'])
                batch_labels.append(result['labels'])
                batch_sids.extend(result['subject_ids'])
                successful += 1
            
            files_in_batch += 1
            
            # Save batch when full OR at the end
            is_last = (file_info == files_remaining[-1])
            if files_in_batch >= batch_size or is_last:
                if batch_epochs:
                    chunk_path = output_dir / f'tuab_{split}_chunk{chunk_idx:02d}.h5'
                    
                    epochs_arr = np.concatenate(batch_epochs, axis=0)
                    labels_arr = np.concatenate(batch_labels, axis=0)
                    
                    with h5py.File(chunk_path, 'w') as f:
                        f.create_dataset('epochs', data=epochs_arr, 
                                       compression='gzip', compression_opts=4)
                        f.create_dataset('labels', data=labels_arr, compression='gzip')
                        f.create_dataset('subject_ids', 
                                       data=np.array(batch_sids, dtype='S20'))
                        
                        f.attrs['sfreq'] = config.SFREQ_TARGET
                        f.attrs['n_channels'] = 19
                        f.attrs['n_samples'] = int(config.EPOCH_DURATION * config.SFREQ_TARGET)
                        f.attrs['epoch_duration'] = 30.0
                        f.attrs['dataset'] = 'tuab'
                        f.attrs['task'] = 'abnormal_detection'
                        f.attrs['n_classes'] = 2
                        f.attrs['class_names'] = str(['Normal', 'Abnormal'])
                        f.attrs['chunk_index'] = chunk_idx
                        f.attrs['files_in_chunk'] = files_in_batch
                        f.attrs['split'] = split
                    
                    size_mb = chunk_path.stat().st_size / 1e6
                    print(f"\n  ✓ Saved chunk {chunk_idx}: {len(epochs_arr)} epochs ({size_mb:.1f} MB)")
                    
                    chunk_idx += 1
                
                # Reset batch
                batch_epochs, batch_labels, batch_sids = [], [], []
                files_in_batch = 0
        
        # Summary
        all_chunks = sorted(output_dir.glob(f'tuab_{split}_chunk*.h5'))
        total_epochs = 0
        for c in all_chunks:
            with h5py.File(c, 'r') as f:
                total_epochs += len(f['epochs'])
        print(f"\n  {split} complete: {len(all_chunks)} chunks, {total_epochs} total epochs")
    
    print("\n" + "=" * 60)
    print("✓ TUAB PREPROCESSING COMPLETE")
    print("=" * 60)

In [8]:
# # RUN PREPROCESSING
# # Option 1: Test with limited files
# # process_tuab(max_files_eval=50, max_files_train=50)

# # Option 2: Process all eval only
# # process_tuab(max_files_eval=None, max_files_train=0)

# # Option 3: Process everything
# # process_tuab()

# print("\n" + "=" * 60)
# print("TUAB PREPROCESSING CODE READY")
# print("=" * 60)
# print("""

# To run preprocessing:

# # Test with 50 files from each split:
# process_tuab(max_files_eval=50, max_files_train=50)

# # Process all eval only:
# process_tuab(max_files_eval=None, max_files_train=0)

# # Process everything (both splits):
# process_tuab()

# Output files:
# - data/processed/tuab_eval_processed.h5
# - data/processed/tuab_train_processed.h5

# These files work directly with the UnifiedEEGDataset class!
# """)

In [9]:
# Eval already saved! Only reprocess train.
process_tuab_robust(max_files_eval=0)

TUAB PREPROCESSING (ROBUST)
  eval: 276 files (150 normal, 126 abnormal)
  train: 2717 files (1371 normal, 1346 abnormal)

Processing train (2717 files)...
  Resuming from file 0, chunk 0
train:   7%|▋         | 200/2717 [06:09<21:00:42, 30.05s/it]
  ✓ Saved chunk 0: 8747 epochs (1853.8 MB)
train:  15%|█▍        | 400/2717 [12:14<19:16:28, 29.95s/it]
  ✓ Saved chunk 1: 8412 epochs (1782.6 MB)
train:  22%|██▏       | 600/2717 [18:32<18:08:20, 30.85s/it]
  ✓ Saved chunk 2: 8636 epochs (1830.0 MB)
train:  29%|██▉       | 800/2717 [24:43<16:36:19, 31.18s/it]
  ✓ Saved chunk 3: 8769 epochs (1858.8 MB)
train:  37%|███▋      | 1000/2717 [31:09<15:08:33, 31.75s/it]
  ✓ Saved chunk 4: 8988 epochs (1905.5 MB)
train:  44%|████▍     | 1200/2717 [37:35<14:01:15, 33.27s/it]
  ✓ Saved chunk 5: 9475 epochs (2008.5 MB)
train:  52%|█████▏    | 1400/2717 [44:15<12:23:13, 33.86s/it]
  ✓ Saved chunk 6: 9127 epochs (1935.7 MB)
train:  59%|█████▉    | 1600/2717 [50:54<10:13:11, 32.94s/it]
  ✓ Saved chunk 7: 

In [10]:
# CHUNK LOADER UTILITY (for training pipeline later)

def load_tuab_chunks(output_dir: Path, split: str = 'train') -> Dict:
    """Load all TUAB chunks into a single dataset."""
    output_dir = Path(output_dir)
    chunk_files = sorted(output_dir.glob(f'tuab_{split}_chunk*.h5'))
    
    if not chunk_files:
        print(f"No chunks found for {split}")
        return None
    
    all_epochs, all_labels, all_sids = [], [], []
    
    for f in chunk_files:
        with h5py.File(f, 'r') as hf:
            all_epochs.append(hf['epochs'][:])
            all_labels.append(hf['labels'][:])
            all_sids.extend([s.decode() for s in hf['subject_ids'][:]])
        print(f"  ✓ {f.name}: {all_epochs[-1].shape[0]} epochs")
    
    combined = {
        'epochs': np.concatenate(all_epochs),
        'labels': np.concatenate(all_labels),
        'subject_ids': all_sids,
        'sfreq': config.SFREQ_TARGET,
        'n_channels': 19,
        'n_samples': int(config.EPOCH_DURATION * config.SFREQ_TARGET)
    }
    
    print(f"\nTotal: {len(combined['epochs'])} epochs")
    print(f"Normal: {sum(combined['labels']==0)}, Abnormal: {sum(combined['labels']==1)}")
    return combined


# Quick verification of what's already saved
print("=== EXISTING FILES ===")
for f in sorted(config.PROCESSED_DIR.glob('tuab_*')):
    try:
        with h5py.File(f, 'r') as hf:
            n = len(hf['epochs'])
            size = f.stat().st_size / 1e6
            print(f"  ✓ {f.name}: {n:,} epochs ({size:.1f} MB)")
    except:
        print(f"  ✗ {f.name}: corrupted")

=== EXISTING FILES ===
  ✓ tuab_eval_processed.h5: 12,241 epochs (2594.5 MB)
  ✓ tuab_train_chunk00.h5: 8,747 epochs (1853.8 MB)
  ✓ tuab_train_chunk01.h5: 8,412 epochs (1782.6 MB)
  ✓ tuab_train_chunk02.h5: 8,636 epochs (1830.0 MB)
  ✓ tuab_train_chunk03.h5: 8,769 epochs (1858.8 MB)
  ✓ tuab_train_chunk04.h5: 8,988 epochs (1905.5 MB)
  ✓ tuab_train_chunk05.h5: 9,475 epochs (2008.5 MB)
  ✓ tuab_train_chunk06.h5: 9,127 epochs (1935.7 MB)
  ✓ tuab_train_chunk07.h5: 8,861 epochs (1877.8 MB)
  ✓ tuab_train_chunk08.h5: 8,864 epochs (1878.6 MB)
  ✓ tuab_train_chunk09.h5: 8,830 epochs (1871.5 MB)
  ✓ tuab_train_chunk10.h5: 8,956 epochs (1897.7 MB)
  ✓ tuab_train_chunk11.h5: 9,122 epochs (1934.1 MB)
  ✓ tuab_train_chunk12.h5: 9,251 epochs (1961.0 MB)
  ✓ tuab_train_chunk13.h5: 7,423 epochs (1575.2 MB)
  ✗ tuab_train_processed.h5: corrupted


In [1]:
from pathlib import Path
corrupted = Path('data/processed/tuab_train_processed.h5')
if corrupted.exists():
    corrupted.unlink()
    print("✓ Removed corrupted tuab_train_processed.h5")

✓ Removed corrupted tuab_train_processed.h5


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>